In [0]:
# A hello world example of merge into feature of delta tables

from pyspark.sql import Row

# Existing data (simulated as the target table)
existing_data = [
    Row(product_id=1, qty=10, amount=100.0),
    Row(product_id=2, qty=5, amount=50.0),
    Row(product_id=3, qty=20, amount=200.0),
]

df_existing = spark.createDataFrame(existing_data)
df_existing.printSchema()
df_existing.show()

root
 |-- product_id: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- amount: double (nullable = true)

+----------+---+------+
|product_id|qty|amount|
+----------+---+------+
|         1| 10| 100.0|
|         2|  5|  50.0|
|         3| 20| 200.0|
+----------+---+------+



In [0]:
# change schema with yours please
TABLE = "aon_impact.gksdb.sales_temp"
df_existing.write.format("delta").mode("overwrite").saveAsTable(TABLE)

In [0]:
%sql

SELECT * FROM aon_impact.gksdb.sales_temp;

product_id,qty,amount
1,10,100.0
2,5,50.0
3,20,200.0


In [0]:
# new data represent newly computed output, like from stream, or from delta live table [discussed later]
# where you want to merge the result, see we don't have complete output. we have some results, we want to either update old table or insert new records
new_data = [
    Row(product_id=2, qty=10, amount=90.0),   # update existing product_id=2
    Row(product_id=4, qty=7, amount=70.0),    # insert new product_id=4
]

df_new = spark.createDataFrame(new_data)

In [0]:
# Merge with dataframe APIs

from delta.tables import DeltaTable

delta_sales = DeltaTable.forName(spark, "aon_impact.gksdb.sales_temp")

(delta_sales.alias("target")
 .merge(df_new.alias("source"), "target.product_id = source.product_id")
 .whenMatchedUpdate(set={
     "qty": "source.qty",
     "amount": "source.amount"
 })
 .whenNotMatchedInsert(values={
     "product_id": "source.product_id",
     "qty": "source.qty",
     "amount": "source.amount"
 })
 .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

SELECT * FROM aon_impact.gksdb.sales_temp;

product_id,qty,amount
1,10,100.0
3,20,200.0
4,7,70.0
2,10,90.0


In [0]:
%sql
MERGE INTO aon_impact.gksdb.sales_temp AS target
USING (
  SELECT 3 AS product_id, 30 AS qty, 120.0 AS amount
  UNION ALL
  SELECT 9 AS product_id, 9, 99.0
) AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN
  UPDATE SET target.qty = source.qty, target.amount = source.amount
WHEN NOT MATCHED THEN
  INSERT (product_id, qty, amount)
  VALUES (source.product_id, source.qty, source.amount)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql

SELECT * FROM aon_impact.gksdb.sales_temp

product_id,qty,amount
1,10,100.0
2,10,90.0
4,7,70.0
3,30,120.0
9,9,99.0
